# Cord Blood Benchmark — Session Progress Report

Generated from `reports/cordblood/build_report.py`. Each section is a thin wrapper.

Sections:
1. Dataset summary
2. Evaluation tasks setup
3. Completion matrix
4. Per-method metrics + leaderboard
5. Training-curve plots
6. UMAP trajectory per method

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('/rds/user/wz369/hpc-work/PINN_dynamics/reports/cordblood')))
import build_report as br
import pandas as pd
from IPython.display import Image, Markdown, display

## 1. Dataset summary

In [ ]:
summary, adata = br.dataset_summary()
summary

## 2. Evaluation tasks setup

Tier 1 (4 methods × 2 embeddings × 3 seeds = 24) + DeepRUOT + scDiffEq (= +12, total 36).
Tier 2 (MIOFlow/TIGON/TrajectoryNet) is deferred.

In [ ]:
tasks = br.eval_tasks_table()
tasks.groupby(['method', 'embedding']).size().unstack(fill_value=0)

## 3. Completion matrix

Which (method, embedding, seed) cells have `eval_combined.csv` right now?

In [ ]:
matrix = br.completion_matrix()
matrix.groupby(['method', 'embedding']).evaluated.sum().unstack(fill_value=0)

In [ ]:
grid_path = br.plot_completion_grid(matrix)
display(Image(str(grid_path)))

## 4. Per-method metrics + leaderboard

In [ ]:
per_seed = br.gather_metrics(matrix)
per_seed

In [ ]:
lb = br.leaderboard(per_seed)
lb

## 5. Training-curve plots

In [ ]:
tc_path = br.plot_training_curves()
display(Image(str(tc_path)))

## 6. UMAP trajectory per method

Background = train cells (grey). Foreground = test cells coloured by `def_lab`. Trajectories overlaid where saved.

In [ ]:
umap_xy = br.get_or_make_umap(adata)
import numpy as np
np.savez(br.DATA_DIR / 'umap_xy.npz', X_umap=umap_xy,
         split=adata.obs['split'].astype(str).values,
         def_lab=adata.obs['def_lab'].astype(str).values,
         timepoint=adata.obs['timepoint_tx_days'].values)
print('Saved UMAP to', br.DATA_DIR / 'umap_xy.npz')

In [ ]:
for m in br.METHODS:
    for e in br.EMBS:
        for s in br.SEEDS:
            if (br.LOG_ROOT / m / f'{e}_s{s}').exists():
                p = br.plot_method_umap(adata, m, e, s, umap_xy=umap_xy)
                display(Markdown(f'**{m} / {e} / s{s}**'))
                display(Image(str(p)))
                break